# Challenge 7 · The Quantum Network Cut
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hastikacheddy/qiskit-fall-fest-participant/blob/main/challenges/challenge_07_qaoa_maxcut/challenge_07_qaoa_maxcut.ipynb)

**Tier 2 - Quantum Algorithms and Applications** · **Estimated time: 45-60 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Split a small network into two data centres so that as many links as possible cross between them (Max-Cut). Compare a brute-force answer with a variational quantum approach (QAOA).

### Learning objectives
- Define the Max-Cut objective and compute it classically.
- Express it as a cost Hamiltonian made of Pauli Z operators.
- Build a QAOA circuit and optimise its parameters with a classical optimiser.
- Separate correctness checks from solution-quality benchmarks.

### Prerequisites
- Challenges 1-3 or equivalent
- Pauli Z and rotation gates
- A basic idea of optimisation (minimising a function)

### What you will write
| Function | Status |
|---|---|
| `cut_value` | required |
| `brute_force_maxcut` | required |
| `cost_hamiltonian` | required |
| `qaoa_circuit` | required |
| `qaoa_expectation` | required |
| `solve_maxcut_qaoa` | required |
| `optimal_cut_probability` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `cut_value` evaluates the (weighted) cut of a bit-string in Qiskit bit order.
- `brute_force_maxcut` returns the exact optimum and all optimal bit-strings.
- `cost_hamiltonian` is the diagonal operator whose eigenvalue for each bit-string equals its cut value.
- `qaoa_circuit` prepares exactly the documented QAOA state (convention fixed in the problem statement).
- `qaoa_expectation` equals <psi|H_C|psi> and is invariant under node relabelling.
- `solve_maxcut_qaoa` is self-consistent: the reported cut value and expectation are recomputed from the returned bit-string and parameters.
- Invalid graphs, bit-strings and parameter lists raise ValueError.

**Quality benchmarks** (reported separately from correctness - a stochastic optimiser is not required to be perfect):
- p=1 QAOA with restarts reaches a good fraction of the best p=1 expectation (benchmark, not correctness).
- The returned bit-string is a high-quality cut relative to the exact optimum (benchmark, not correctness).

**Bonus (optional, never required to pass):**
- `optimal_cut_probability` is the exact probability that one measurement of the QAOA state returns an optimal cut (every bit-string that reaches the maximum cut, ties included): the share of optimal strings at zero parameters, unchanged by relabelling the nodes.

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

# Colab opens a notebook straight from GitHub without the project next to it: the project is fetched from this repository.
PROJECT_REPOSITORY = 'https://github.com/hastikacheddy/qiskit-fall-fest-participant.git'

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    try:
        on_colab = importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:  # there is no `google` package at all
        on_colab = False
    if root is None and on_colab:
        # the notebook is here, the project is not: fetch it (a few seconds)
        root = here / PROJECT_REPOSITORY.rsplit('/', 1)[-1].removesuffix('.git')
        if not root.exists():
            print('Fetching the project from', PROJECT_REPOSITORY, '...')
            try:
                subprocess.check_call(['git', 'clone', '--quiet', '--depth', '1', PROJECT_REPOSITORY, str(root)])
            except (subprocess.CalledProcessError, OSError) as error:
                raise RuntimeError(f'Could not fetch the project from {PROJECT_REPOSITORY} ({error}). '
                                   "Upload the project yourself instead (see README.md, section 'Colab').") from None
    if root is None or not (root / 'pyproject.toml').is_file():
        raise RuntimeError('Open this notebook from inside the project folder (the one that contains pyproject.toml). '
                           'If you only have the notebook, download the whole project from ' + PROJECT_REPOSITORY.removesuffix('.git') +
                           " (Code > Download ZIP) or open the notebook in Google Colab - see README.md, section 'Colab'.")
    print('Installing the project and its libraries - about a minute the first time ...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
A company runs a network of **data centres** connected by links; every link carries traffic with a known weight. The network is to be split into **two groups** of data centres. The traffic of all links *between* the groups is what the split **cuts**. Find a split that cuts as much traffic as possible - the **Max-Cut** problem. You will

1. compute the cut classically (`cut_value`, `brute_force_maxcut`),
2. write the problem as a **cost Hamiltonian** (`cost_hamiltonian`),
3. build the **QAOA** circuit (`qaoa_circuit`) and its exact energy (`qaoa_expectation`),
4. let a classical optimiser tune the circuit parameters (`solve_maxcut_qaoa`) and judge the result against the exact optimum.
5. *(Bonus)* ask how likely it is to *sample* an optimal cut, not just how good the average is (`optimal_cut_probability`).

### Conventions (used by the notebook and by every test)
* A **graph** is `(num_nodes, edges)`: nodes are `0 … num_nodes − 1`, `edges` is a list of `(u, v, weight)` triples (weights are finite real numbers; an undirected edge appears once; no self-loops). The helper `validate_graph(num_nodes, edges)` is provided: it raises `ValueError` for invalid graphs and returns the clean edge list.
* **Node `u` is qubit `u`.** A **bit-string** `z` is written in Qiskit order, so node `u` is the character `u` **from the right**: `z_u = z[n − 1 − u]`. The bit tells the group (0 or 1) of the node.
* **Cut value:** C(z) = Σ over edges (u, v, w) of w · [z_u ≠ z_v].
* **Cost Hamiltonian:** H_C = Σ w · (I − Z_u Z_v) / 2. Its eigenvalue on the basis state |z⟩ is exactly C(z); we *maximise* ⟨H_C⟩.
* **QAOA state with p layers** and parameters γ₁ … γ_p, β₁ … β_p:
  |γ, β⟩ = U_B(β_p) U_C(γ_p) ⋯ U_B(β₁) U_C(γ₁) · H^{⊗n} |0…0⟩,  with  U_C(γ) = exp(−i γ H_C)  and  U_B(β) = exp(−i β Σ_u X_u).
  Layer 1 is applied first; inside a layer the cost operator comes before the mixer. Global phases do not matter.

### Function contracts
| Function | Input | Output | Raises `ValueError` when |
|---|---|---|---|
| `cut_value(bitstring, edges)` | bit-string (its length is the number of nodes), edge list | `float` C(z) | bit-string empty / not 0-1, invalid edges (endpoints must be nodes of the bit-string) |
| `brute_force_maxcut(num_nodes, edges)` | graph with 1 … 16 nodes | `(best_value, optimal_strings)`: the maximum cut (`float`) and **all** bit-strings reaching it (within 1e-9), sorted as strings | invalid graph, more than 16 nodes |
| `cost_hamiltonian(num_nodes, edges)` | graph | `qiskit.quantum_info.SparsePauliOp` on `num_nodes` qubits with `diag(H_C)[int(z, 2)] = C(z)` | invalid graph |
| `qaoa_circuit(num_nodes, edges, gammas, betas)` | graph, two equally long lists of floats (p ≥ 0 layers) | `QuantumCircuit` on `num_nodes` qubits **without measurements**, preparing the QAOA state above (any gate decomposition) | invalid graph, unequal lengths, non-finite parameters |
| `qaoa_expectation(num_nodes, edges, gammas, betas)` | as above | `float` ⟨γ,β\|H_C\|γ,β⟩, exact | as above |
| `solve_maxcut_qaoa(num_nodes, edges, p=1, restarts=5, seed=0)` | graph, depth p ≥ 1, number of random restarts ≥ 1, integer seed ≥ 0 | `dict` with `gammas`, `betas` (lists of length p), `expectation` (= ⟨H_C⟩ at these parameters), `bitstring` (the **most probable** bit-string of the final state; among bit-strings within 1e-9 of the largest probability the one with the smaller integer value), `cut_value` (= C of that bit-string). Same arguments → same result | invalid graph, `p` / `restarts` not a positive integer, `seed` not a non-negative integer |
| *(bonus)* `optimal_cut_probability(num_nodes, edges, gammas, betas)` | as for `qaoa_expectation` | `float` in [0, 1]: the exact total probability of **all optimal bit-strings** (those reaching the maximum cut, within 1e-9 - the strings `brute_force_maxcut` lists) in the QAOA state | as for `qaoa_expectation` |

**Correctness versus quality.** The correctness tests are exact (tolerance 1e-9) and never depend on how good your optimiser is: the returned numbers must be *consistent* (the reported expectation is the expectation of the returned parameters, the reported cut is the cut of the returned bit-string, nothing exceeds the true optimum). **Quality benchmarks** are reported separately and are *not* part of correctness: with `p = 1` and `restarts = 15` (i) the returned expectation reaches at least 95 % of the best value of a dense (γ, β) grid, (ii) the returned cut has at least 80 % of the optimal cut weight. (The landscape has several local maxima; a single start finds the best one only about 40 % of the time on the example graph - that is what the restarts are for.)

## Concept: from a graph to a circuit
* **Max-Cut is hard** (NP-hard); for the five-node example brute force is trivial (2⁵ = 32 splits), but the number of splits doubles with every node.
* **A cut as an energy.** With Z|0⟩ = +|0⟩ and Z|1⟩ = −|1⟩ the product Z_uZ_v is +1 for equal bits and −1 for different bits, so (1 − Z_uZ_v)/2 is 1 exactly when the edge is cut. Summing over edges gives the **cost Hamiltonian** whose energy is the cut value.
* **QAOA** alternates two operations. The *cost layer* exp(−iγH_C) puts a phase on every bit-string that depends on its cut value; the *mixer* exp(−iβΣX) moves probability between neighbouring bit-strings. Starting from the uniform superposition, good parameters (γ, β) concentrate probability on large cuts.
* **Classical outer loop.** A classical optimiser changes (γ, β) to maximise the exact expectation ⟨H_C⟩. With p layers there are 2p parameters; more layers can only help in principle (and cost more to optimise).
* **Approximation ratio** ⟨H_C⟩ / C_max measures how close the *average* outcome is to the optimum; picking the **most probable** bit-string can do better than the average.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize
from qiskit_fest.challenges.c07_qaoa_maxcut import validate_graph, validate_parameters

In [ ]:
from qiskit_fest.challenges.c07_qaoa_maxcut import (
    EXAMPLE_EDGES, EXAMPLE_NUM_NODES, EXAMPLE_OPTIMUM, plot_cut_histogram, plot_graph, plot_landscape)

plot_graph(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, None, "The example network: 5 data centres, 6 links")
print("edges (u, v, weight):", EXAMPLE_EDGES)
print("classical optimum of this graph (given, you will compute it yourself in Part 2):", EXAMPLE_OPTIMUM)

## Part 1 · Evaluate a cut  *(TODO)*
Complete `cut_value(bitstring, edges)`: the total weight of the edges whose two endpoints lie in different groups. **Remember the bit order:** node `u` is the character `u` from the *right*.

<details><summary>💡 Hint 1 (subtle)</summary>

Which character of the string belongs to node 0? To node `n − 1`? A reversed string describes a *different* split - and for symmetric strings you would not notice.
</details>

<details><summary>💡 Hint 2</summary>

First turn the string into a list `side` with `side[u]` = group of node `u`. An edge `(u, v, w)` is cut exactly if `side[u] != side[v]`; add `w` for those edges. The number of nodes is `len(bitstring)`; call the provided `validate_graph(n, edges)` first (it also checks that the edges fit the bit-string).
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

```python
n = len(bitstring)
edges = validate_graph(n, edges)
side = [int(bitstring[___]) for u in range(n)]          # index counted from the right
return float(sum(w for u, v, w in edges if ___))
```
Do not forget to reject an empty string or characters other than 0 and 1 (`ValueError`).
</details>

In [ ]:
def cut_value(bitstring: str, edges) -> float:
    """Weighted cut of `bitstring` (Qiskit order: node u is the character u from the right); edges are (u, v, weight)."""
    # TODO: validate the bit-string and the edges (ValueError), then add up the weights of the cut edges.
    raise NotImplementedError("TODO: implement cut_value()")

In [ ]:
with if_ready():
    for z in ("00001", "00110", "01100"):
        print(f"cut({z}) = {cut_value(z, EXAMPLE_EDGES)}")
    plot_graph(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, "00110", "Split 00110: cut weight %g" % cut_value("00110", EXAMPLE_EDGES))

In [ ]:
report = run_public_checks(7, globals(), keyword="cut_value")

## Part 2 · The exact answer by brute force  *(TODO)*
Complete `brute_force_maxcut(num_nodes, edges)`: try all 2ⁿ splits and return the best value **and all optimal bit-strings** (a split and its complement always have the same cut, so there are at least two - unless there are no edges).

<details><summary>💡 Hint 1 (subtle)</summary>

`format(i, f"0{n}b")` turns the integer `i` into an n-bit string. Which strings reach the maximum - exactly one, or several?
</details>

<details><summary>💡 Hint 2</summary>

Collect the value of every string first, then keep all strings whose value is within 1e-9 of the maximum (weights may be floats). Reject more than 16 nodes (`ValueError`).
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Loop `for i in range(2**n)`, use your `cut_value`, track `best`; afterwards `[z for z, v in zip(strings, values) if v >= best - 1e-9]`. A vectorised NumPy version over all integers `i` (bit `u` of `i` is node `u`) is much faster for n ≥ 12.
</details>

In [ ]:
def brute_force_maxcut(num_nodes: int, edges) -> tuple[float, list[str]]:
    """(best cut value, all optimal bit-strings sorted ascending) of a graph with at most 16 nodes."""
    # TODO: validate (ValueError), evaluate every bit-string, return the maximum and all maximisers.
    raise NotImplementedError("TODO: implement brute_force_maxcut()")

In [ ]:
with if_ready():
    best, optimal = brute_force_maxcut(EXAMPLE_NUM_NODES, EXAMPLE_EDGES)
    print("maximum cut:", best, "  optimal bit-strings:", optimal)
    plot_graph(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, optimal[0], f"An optimal split {optimal[0]}: cut weight {best:g}")

In [ ]:
report = run_public_checks(7, globals(), keyword="brute_force")

## Part 3 · The cost Hamiltonian  *(TODO)*
Complete `cost_hamiltonian(num_nodes, edges)` so that the diagonal of its matrix lists the cut values: `H_C = Σ w (I − Z_u Z_v)/2`. Return a `SparsePauliOp` (identity term included).

<details><summary>💡 Hint 1 (subtle)</summary>

Z has eigenvalue +1 on |0⟩ and −1 on |1⟩. What is the eigenvalue of Z_u Z_v on a basis state whose bits `u` and `v` are equal? different? Which combination of the identity and Z_u Z_v is 0 in the first case and 1 in the second?
</details>

<details><summary>💡 Hint 2</summary>

Each edge contributes `w/2` times the identity and `−w/2` times Z_u Z_v. A Pauli label is written in Qiskit order too: the **rightmost** character acts on qubit 0, so `"ZIZ"` means Z on qubits 0 and 2.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Start with the label `"I" * n` and coefficient `0.5 * sum(weights)`. For each edge build a list of `n` letters `"I"`, put `"Z"` at the positions `n − 1 − u` and `n − 1 − v`, join it and add it with coefficient `-0.5 * w`. `SparsePauliOp(labels, coeffs)`; `.simplify()` is optional.
</details>

In [ ]:
def cost_hamiltonian(num_nodes: int, edges) -> SparsePauliOp:
    """H_C = sum over edges of w (I - Z_u Z_v) / 2 as a SparsePauliOp on num_nodes qubits."""
    # TODO: validate (ValueError), collect Pauli labels and coefficients, return the SparsePauliOp.
    raise NotImplementedError("TODO: implement cost_hamiltonian()")

In [ ]:
with if_ready():
    H = cost_hamiltonian(EXAMPLE_NUM_NODES, EXAMPLE_EDGES)
    print(H)
    diagonal = np.real(np.diag(H.to_matrix()))
    print("eigenvalue on |00110> =", diagonal[int("00110", 2)], " (cut value of 00110 is", cut_value("00110", EXAMPLE_EDGES), ")")
    print("largest eigenvalue =", diagonal.max(), "= the maximum cut")

In [ ]:
report = run_public_checks(7, globals(), keyword="cost_hamiltonian")

## Part 4 · The QAOA circuit  *(TODO)*
Complete `qaoa_circuit(num_nodes, edges, gammas, betas)` for the state defined in the conventions: Hadamards on all qubits, then for every layer *k* the cost operator exp(−iγ_k H_C) followed by the mixer exp(−iβ_k Σ X). No measurements.

<details><summary>💡 Hint 1 (subtle)</summary>

H_C is a sum of commuting terms, so exp(−iγH_C) is a *product* with one factor per edge, and up to a global phase only the Z⊗Z part matters. The mixer is a product of single-qubit rotations.
</details>

<details><summary>💡 Hint 2</summary>

In Qiskit `qc.rzz(θ, u, v)` is exp(−i θ/2 · Z_u Z_v) and `qc.rx(φ, q)` is exp(−i φ/2 · X_q). Compare the exponents with exp(−iγ H_C) and exp(−iβ X): the cost part contains −(w/2) Z_u Z_v, so watch the **sign** and the **factor 2** of the angles.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`qc.h(range(n))`; for each layer: for every edge `qc.rzz(-gamma * w, u, v)`, then `qc.rx(2 * beta, range(n))`. Validate with the provided `validate_graph` and `validate_parameters(gammas, betas)` (they raise `ValueError`).
</details>

In [ ]:
def qaoa_circuit(num_nodes: int, edges, gammas, betas) -> QuantumCircuit:
    """QAOA state preparation: H on all qubits, then p layers of exp(-i gamma H_C) and exp(-i beta sum X). No measurements."""
    # TODO: validate graph and parameters (ValueError), build the circuit layer by layer.
    raise NotImplementedError("TODO: implement qaoa_circuit()")

In [ ]:
with if_ready():
    from qiskit_fest.viz import draw_circuit

    draw_circuit(qaoa_circuit(3, [(0, 1, 1.0), (1, 2, 1.0)], [0.4], [0.3]), "QAOA, p = 1, on a path of three nodes")

In [ ]:
report = run_public_checks(7, globals(), keyword="qaoa_circuit")

## Part 5 · The energy of the QAOA state  *(TODO)*
Complete `qaoa_expectation(num_nodes, edges, gammas, betas)`: the exact value ⟨γ,β|H_C|γ,β⟩. It is the number the optimiser will maximise. Sanity values: with all parameters 0 it is half the total weight (a random split); it can never exceed the maximum cut.

<details><summary>💡 Hint</summary>

Simulate your circuit exactly with `Statevector(circuit)` and use `.expectation_value(operator)` with your cost Hamiltonian; take the real part.
</details>

In [ ]:
def qaoa_expectation(num_nodes: int, edges, gammas, betas) -> float:
    """Exact <H_C> of the QAOA state (real number)."""
    # TODO: build the circuit and the Hamiltonian, simulate exactly, return the real expectation value.
    raise NotImplementedError("TODO: implement qaoa_expectation()")

In [ ]:
with if_ready():
    print("random split (all parameters 0):", qaoa_expectation(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, [0.0], [0.0]), "= half the total weight")
    plot_landscape(qaoa_expectation, EXAMPLE_NUM_NODES, EXAMPLE_EDGES, optimum=EXAMPLE_OPTIMUM)

In [ ]:
report = run_public_checks(7, globals(), keyword="qaoa_expectation")

## Part 6 · Let the classical optimiser tune the circuit  *(TODO)*
Complete `solve_maxcut_qaoa(num_nodes, edges, p=1, restarts=5, seed=0)`: maximise ⟨H_C⟩ over the 2p parameters with a classical optimiser, restarting from several random points, and return a dictionary with the best parameters, their expectation, the **most probable bit-string** of the final state and its cut value.

<details><summary>💡 Hint 1 (subtle)</summary>

Optimisers *minimise*. What do you minimise to maximise ⟨H_C⟩? The landscape has several local optima - one start may not be enough, which is why there are restarts.
</details>

<details><summary>💡 Hint 2</summary>

`scipy.optimize.minimize(objective, x0, method="COBYLA")` with `x = [γ₁ … γ_p, β₁ … β_p]`. Draw the starting points from `rng = np.random.default_rng(seed)`: γ uniformly from [0, π], β from [0, π/2], so equal arguments give equal results. Keep the result with the best value.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

After optimising, simulate the final circuit, take `probabilities = Statevector(circuit).probabilities()`, pick the index of the largest entry (the smallest index among entries within 1e-9 of the maximum), format it with `format(index, f"0{n}b")` and evaluate it with **your** `cut_value`. Report `expectation` with your `qaoa_expectation` at the final parameters.
</details>

In [ ]:
def solve_maxcut_qaoa(num_nodes: int, edges, p: int = 1, restarts: int = 5, seed: int = 0) -> dict:
    """Optimise a p-layer QAOA: dict with gammas, betas, expectation, bitstring (most probable) and cut_value."""
    # TODO: validate, optimise -<H_C> from `restarts` seeded random starting points, report consistent numbers.
    raise NotImplementedError("TODO: implement solve_maxcut_qaoa()")

In [ ]:
with if_ready():
    result = solve_maxcut_qaoa(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, p=1, restarts=5, seed=3)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in result.items() if k not in ("gammas", "betas")})
    print("approximation ratio <H_C> / optimum =", round(result["expectation"] / EXAMPLE_OPTIMUM, 3), " | random split:", round(3.0 / EXAMPLE_OPTIMUM, 3))
    state = Statevector(qaoa_circuit(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, result["gammas"], result["betas"]))
    plot_cut_histogram(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, state.probabilities_dict(), cut_value)
    plot_graph(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, result["bitstring"], f"QAOA's answer {result['bitstring']}: cut weight {result['cut_value']:g}")

In [ ]:
report = run_public_checks(7, globals(), keyword="solve_maxcut_qaoa")

## Part 7 · How good is QAOA?  *(not graded)*
Compare p = 1, 2, 3 with the optimum and with a random split. Then try your own graphs (ring, complete graph, weighted edges).

In [ ]:
with if_ready():
    best, optimal = brute_force_maxcut(EXAMPLE_NUM_NODES, EXAMPLE_EDGES)
    print(f"optimum {best:g}   random split {0.5 * sum(w for *_, w in EXAMPLE_EDGES):g}")
    print(f"{'p':>2} {'<H_C>':>8} {'ratio':>7} {'QAOA answer':>12} {'its cut':>8}")
    for p in (1, 2, 3):
        r = solve_maxcut_qaoa(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, p=p, restarts=4, seed=1)
        print(f"{p:>2} {r['expectation']:>8.3f} {r['expectation'] / best:>7.3f} {r['bitstring']:>12} {r['cut_value']:>8g}")

## Bonus · The probability of sampling an optimal cut  *(optional)*
On real hardware you never see the exact state - you see **samples**. What matters then is not the average ⟨H_C⟩ but the chance that *one measurement* returns an optimal cut.
Complete `optimal_cut_probability(num_nodes, edges, gammas, betas)`: the **exact** total probability of all optimal bit-strings (those reaching the maximum cut, within 1e-9 - exactly the strings `brute_force_maxcut` lists) in the QAOA state. It is a number between 0 and 1. With all parameters 0 (or no layers) the state is the uniform superposition, so it equals (number of optimal strings) / 2ⁿ.

<details><summary>💡 Hint 1 (subtle)</summary>

Which bit-strings count as *optimal*? You have already written the function that lists them. And which call gives the probability of **every** bit-string of a state at once - and at which index does a bit-string sit?
</details>

<details><summary>💡 Hint 2 (explicit)</summary>

`Statevector(circuit).probabilities()` is indexed by the integer value of the bit-string (`int(z, 2)`). Add up the entries of the optimal strings. Use your own `brute_force_maxcut`, `qaoa_circuit` (and the provided validation through them) so that invalid input raises `ValueError`.
</details>

In [ ]:
def optimal_cut_probability(num_nodes: int, edges, gammas, betas) -> float:
    """Exact probability that measuring the QAOA state returns an optimal cut (any bit-string reaching the maximum cut)."""
    # TODO (bonus): list the optimal bit-strings, simulate the circuit exactly, add up their probabilities.
    raise NotImplementedError("TODO (bonus): implement optimal_cut_probability()")

In [ ]:
with if_ready():
    best, optimal = brute_force_maxcut(EXAMPLE_NUM_NODES, EXAMPLE_EDGES)
    tuned = solve_maxcut_qaoa(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, p=1, restarts=8, seed=1)
    uniform_probability = optimal_cut_probability(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, [0.0], [0.0])
    tuned_probability = optimal_cut_probability(EXAMPLE_NUM_NODES, EXAMPLE_EDGES, tuned["gammas"], tuned["betas"])
    print(f"optimal cuts of the example: {optimal}  ({len(optimal)} of {2 ** EXAMPLE_NUM_NODES} bit-strings)")
    print(f"probability that one measurement is optimal - uniform state (all parameters 0): {uniform_probability:.3f} = {len(optimal)}/{2 ** EXAMPLE_NUM_NODES}")
    print(f"probability that one measurement is optimal - tuned p = 1 state: {tuned_probability:.3f}")
    print(f"approximation ratio <H_C> / optimum of the same state: {tuned['expectation'] / best:.3f}  (the average being good is not the same as an optimal sample being likely)")

In [ ]:
report = run_public_checks(7, globals(), keyword="optimal_cut_probability")

## Reflection
1. Your `cut_value` and the Hamiltonian disagree for the string `01100` in the example. What are the two most likely bugs, and how would you find out which function is wrong?
2. Why does every graph with at least one edge have at least two optimal bit-strings? What does that mean for the *most probable* bit-string of a QAOA state?
3. At γ = 0 or β = 0 the expectation is exactly half the total weight. Why? What does it tell you about the landscape you plotted?
4. The approximation ratio ⟨H_C⟩/C_max for p = 1 is well below 1, yet the most probable bit-string is often an optimal cut. How can both be true? What would you do with *samples* instead of the exact state on real hardware?
5. How does the cost of your classical simulation grow with the number of nodes, and what does that mean for using QAOA beyond ~30 qubits on a simulator?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 7)